## Data Preparation

### Import Library


In [1]:
from dotenv import load_dotenv
import joblib
import matplotlib.pyplot as plt
import mne
import numpy as np
import os
from pathlib import Path
import shutil
from utils.utils import ch_rename

load_dotenv(dotenv_path='.env')

True

### Load dataset

In [2]:
BASE_PATH = os.getenv("BASE_PATH")
DATA_PATH = BASE_PATH + os.getenv("DATA_PATH")

In [3]:
DATASET_DIR = Path(DATA_PATH)
CLEAN_DIR = Path(BASE_PATH + '/cleaned/')
CLEAN_DIR.mkdir(exist_ok=True)

for SUBJECT_DIR in DATASET_DIR.iterdir():
    print(SUBJECT_DIR)
    if SUBJECT_DIR.is_dir():
        for infix in ['R01', 'R02']:
            edf_file = SUBJECT_DIR / f"{SUBJECT_DIR.name}{infix}.edf"
            event_file = SUBJECT_DIR / f"{SUBJECT_DIR.name}{infix}.edf.event"
            for file in [edf_file, event_file]:
                if file.exists():
                    shutil.copy(file, CLEAN_DIR / file.name)
                else:
                    print(f"File {file} does not exist")

Dataset/files/64_channel_sharbrough-old.png
Dataset/files/64_channel_sharbrough.pdf
Dataset/files/64_channel_sharbrough.png
Dataset/files/ANNOTATORS
Dataset/files/RECORDS
Dataset/files/S001
Dataset/files/S002
Dataset/files/S003
Dataset/files/S004
Dataset/files/S005
Dataset/files/S006
Dataset/files/S007
Dataset/files/S008
Dataset/files/S009
Dataset/files/S010
Dataset/files/S011
Dataset/files/S012
Dataset/files/S013
Dataset/files/S014
Dataset/files/S015
Dataset/files/S016
Dataset/files/S017
Dataset/files/S018
Dataset/files/S019
Dataset/files/S020
Dataset/files/S021
Dataset/files/S022
Dataset/files/S023
Dataset/files/S024
Dataset/files/S025
Dataset/files/S026
Dataset/files/S027
Dataset/files/S028
Dataset/files/S029
Dataset/files/S030
Dataset/files/S031
Dataset/files/S032
Dataset/files/S033
Dataset/files/S034
Dataset/files/S035
Dataset/files/S036
Dataset/files/S037
Dataset/files/S038
Dataset/files/S039
Dataset/files/S040
Dataset/files/S041
Dataset/files/S042
Dataset/files/S043
Dataset/file

In [3]:
BASE_PATH = os.getenv("BASE_PATH")
CLEANED_PATH = BASE_PATH + os.getenv("CLEAN_PATH")

eo_list = sorted([f for f in os.listdir(CLEANED_PATH) if f.endswith('R01.edf')])
ec_list = sorted([f for f in os.listdir(CLEANED_PATH) if f.endswith('R02.edf')])

In [4]:
eo_raw = []
ec_raw = []

for fname in eo_list:
    raw = ch_rename(CLEANED_PATH, fname)
    eo_raw.append(raw)

for fname in ec_list:
    raw = ch_rename(CLEANED_PATH, fname)
    ec_raw.append(raw)

In [5]:
eo_crop = []
ec_crop = []

for i in range(len(eo_raw)):
    eo = eo_raw[i].copy()
    if eo.duration > 60:
        eo.crop(tmin=0.00, tmax=60.00, include_tmax=False)
    eo_crop.append(eo)

for i in range(len(ec_raw)):
    ec = ec_raw[i].copy()
    if ec.duration > 60:
        ec.crop(tmin=0.00, tmax=60.00, include_tmax=False)
    ec_crop.append(ec)

In [6]:
PICKLE_PATH = BASE_PATH + os.getenv("PICKLE_PATH")
PICKLE_DIR = Path(PICKLE_PATH)
PICKLE_DIR.mkdir(exist_ok=True)

joblib.dump(eo_crop, PICKLE_DIR / 'eo_crop.pkl')
joblib.dump(ec_crop, PICKLE_DIR / 'ec_crop.pkl')

['Dataset/pickled/ec_crop.pkl']

### Segmentation

In [ ]:
def sliding_windows(data, window_size, stride):
    """
    data: (n_channels, n_samples)
    returns: (n_windows, n_channels, n_samples)
    """

    n_channels, n_samples = data.shape
    n_windows = (n_samples - window_size) // stride + 1
    windows = np.zeros((n_windows, n_channels, window_size))
    for i in range(n_windows):
        windows[i] = data[:, i*stride:i*stride+window_size]
    return windows

In [ ]:
# windows = sliding_windows(data, 1000, 500)
# labels = np.full(len(windows), folder_name)

In [ ]:
import gc

WINDOW_SIZE = 1000
STRIDE = 500
SEGMENT_DIR = 'data/segmented'
os.makedirs(SEGMENT_DIR, exist_ok=True)

# Pass 1: count total windows and get n_channels (no full data load)
file_list = []
total_windows = 0
n_channels = None
for folder_name in sorted(os.listdir(DATASET_DIR)):
    folder_path = os.path.join(DATASET_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue
    for file_name in sorted(os.listdir(folder_path)):
        if not file_name.endswith('.edf'):
            continue
        file_path = os.path.join(folder_path, file_name)
        raw = mne.io.read_raw_edf(file_path, preload=False, verbose=False)
        n_samples = raw.n_times
        if n_channels is None:
            n_channels = raw.info['nchan']
        raw = None
        gc.collect()
        n_win = (n_samples - WINDOW_SIZE) // STRIDE + 1
        total_windows += n_win
        file_list.append((folder_name, file_path, n_win))

unique_subjects = sorted(set(f[0] for f in file_list))
label_to_int = {s: i for i, s in enumerate(unique_subjects)}
print('Total windows:', total_windows, '| n_channels:', n_channels, '| subjects:', len(unique_subjects)) 

In [ ]:
# Pass 2: stream each file into pre-allocated memmap (one file in RAM at a time)
X_path = os.path.join(SEGMENT_DIR, 'X.dat')
y_path = os.path.join(SEGMENT_DIR, 'y.dat')
X_mem = np.memmap(X_path, dtype='float32', mode='w+', shape=(total_windows, n_channels, WINDOW_SIZE))
y_mem = np.memmap(y_path, dtype='int32', mode='w+', shape=(total_windows,))

row = 0
for folder_name, file_path, n_win in file_list:
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    data = raw.get_data()
    win = sliding_windows(data, WINDOW_SIZE, STRIDE).astype(np.float32)
    lab = label_to_int[folder_name]
    X_mem[row:row + n_win] = win
    y_mem[row:row + n_win] = lab
    row += n_win
    del raw, data, win
    gc.collect()

# Views for the rest of the notebook (read-only; data stays on disk)
X = np.memmap(X_path, dtype='float32', mode='r', shape=(total_windows, n_channels, WINDOW_SIZE))
y = np.memmap(y_path, dtype='int32', mode='r', shape=(total_windows,))
print('X shape:', X.shape, '| y shape:', y.shape)

### Save to pickle

In [ ]:
import pickle

meta = {
    'X_path': X_path,
    'y_path': y_path,
    'X_shape': X.shape,
    'y_shape': y.shape,
    'label_to_int': label_to_int,
    'n_channels': n_channels,
    'window_size': WINDOW_SIZE,
    'stride': STRIDE,
}
with open(os.path.join(SEGMENT_DIR, 'segments_meta.pickle'), 'wb') as f:
    pickle.dump(meta, f)
print('Saved to', os.path.join(SEGMENT_DIR, 'segments_meta.pickle'))
print('To load: meta = pickle.load(...); X = np.memmap(meta["X_path"], dtype="float32", mode="r", shape=meta["X_shape"])')